# Middleware — 中间件与回调对标文档：LangChain Core Components > Callbacks

In [ ]:
from langchain_learning.llm import get_llm
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time

llm = get_llm()

**术语：**
- **Middleware（中间件）** = 在 LLM 调用前后插入的自定义逻辑
- **Callback（回调）** = 在特定事件发生时触发的函数，是实现中间件的主要方式
- **事件类型**：开始(on_start)、结束(on_end)、报错(on_error)、新 token(on_llm_new_token)

### 1. 日志回调——记录每次调用

In [ ]:
class LoggingHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print(f"[LLM 开始] 输入: {prompts[0][:50]}...")

    def on_llm_end(self, response, **kwargs):
        print(f"[LLM 结束] 输出: {response.generations[0][0].text[:50]}...")

    def on_llm_error(self, error, **kwargs):
        print(f"[LLM 错误] {error}")

logger = LoggingHandler()
llm.invoke("用一句话介绍 Python", config={"callbacks": [logger]})

**术语：**
- **BaseCallbackHandler** = 回调基类，继承它并覆写需要的方法
- **on_llm_start / on_llm_end** = LLM 开始/结束时触发
- **config={"callbacks": [...]}** = 把回调传给本次调用

### 2. 耗时统计——测量每次调用的时间

In [ ]:
class TimingHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.start = time.time()

    def on_llm_end(self, response, **kwargs):
        elapsed = time.time() - self.start
        print(f"耗时: {elapsed:.2f}s")

timmer = TimingHandler()
llm.invoke("请写一段 100 字左右的短文", config={"callbacks": [timmer]})

**术语：**
- **start / end** 配对使用，可以计算耗时、统计 token 数等
- 所有回调都支持这种开始到结束的模式

### 3. 链级别回调——监控整个链

In [ ]:
class ChainMonitor(BaseCallbackHandler):
    def on_chain_start(self, serialized, inputs, **kwargs):
        print(f"[链开始] 输入: {inputs}")

    def on_chain_end(self, outputs, **kwargs):
        print(f"[链结束] 输出: {str(outputs)[:50]}...")

chain = ChatPromptTemplate.from_template("解释{topic}是什么") | llm | StrOutputParser()
monitor = ChainMonitor()

result = chain.invoke({"topic": "回调函数"}, config={"callbacks": [monitor]})
print("\n最终结果:", result[:80])

**术语：**
- **on_chain_start / on_chain_end** = 整个链（Chain）开始/结束时的回调
- 链回调可以监控整条链的执行，而 LLM 回调只监控 LLM 部分
- 回调可以挂在不同层级：LLM 级别、Chain 级别、全局级别

### 4. 全局回调——所有调用自动生效

In [ ]:
from langchain_core.callbacks import CallbackManager

class GlobalLogger(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        model = serialized.get("kwargs", {}).get("model", "unknown")
        print(f"[全局] 调用模型: {model}")

manager = CallbackManager([GlobalLogger()])
llm_with_callbacks = get_llm().bind(callback_manager=manager)

llm_with_callbacks.invoke("Hello")
llm_with_callbacks.invoke("Hi again")

**术语：**
- **CallbackManager** = 回调管理器，把多个回调组合在一起
- 全局回调 = 对所有 LLM 调用自动生效，不需要每次传 config
- 实际项目中也可以把回调设置在 get_llm() 中

### 5. 实战：组合多个回调

In [ ]:
class TimingHandler(BaseCallbackHandler):
    """记录耗时"""
    def on_llm_start(self, *args, **kwargs):
        self.start = time.time()
    def on_llm_end(self, *args, **kwargs):
        print(f"  ⏱ 耗时: {time.time() - self.start:.2f}s")

class TokenCountHandler(BaseCallbackHandler):
    """统计 token 数"""
    def __init__(self):
        self.count = 0
    def on_llm_new_token(self, token: str, **kwargs):
        self.count += 1
    def on_llm_end(self, *args, **kwargs):
        print(f"  📝 Token 数: {self.count}")

multi = CallbackManager([TimingHandler(), TokenCountHandler()])

print("组合回调输出：")
llm.invoke("请用 50 字介绍量子计算", config={"callbacks": [multi]})

**术语：**
- 多个回调可以同时使用，各自监听不同的事件
- CallbackManager 把多个回调组合成一个传给 config
- 实际项目中常用 LangSmith 做追踪（自带日志、耗时、token 统计）